In [1]:
import pandas as pd
import os

# Set the dataset path
data_path = r"C:\Users\pparr\Documents\Henallux\Henallux PP 2025_2026\Semester 2\Systèmes_intelligents\Dataset\Car Brand Classification Dataset\train"

# List files in the directory
files = os.listdir(data_path)
print("Files in directory:")
print(files)


Files in directory:
['Acura', 'Aston Martin', 'Audi', 'Bentley', 'BMW', 'Buick', 'Cadillac', 'Chevrolet', 'Chrysler', 'Dodge', 'FIAT', 'Ford', 'GMC', 'Honda', 'Hyundai', 'INFINITI', 'Jaguar', 'Jeep', 'Kia', 'Land Rover', 'Lexus', 'Lincoln', 'Mazda', 'Mercedes-Benz', 'MINI', 'Mitsubishi', 'Nissan', 'Porsche', 'Ram', 'Subaru', 'Toyota', 'Volkswagen', 'Volvo']


In [2]:
import pandas as pd
import os

# Set the dataset path
data_path = r"C:\Users\pparr\Documents\Henallux\Henallux PP 2025_2026\Semester 2\Systèmes_intelligents\Dataset\training-car\training_set"

# List files in the directory
files = os.listdir(data_path)
print("Files in directory:")
print(files)


Files in directory:
['Audi', 'Bentley', 'BMW', 'Chevrolet', 'Chrysler', 'Dodge', 'Ferrari', 'Ford', 'GMC', 'Honda', 'Hyundai', 'Lamborghini', 'Mercedes', 'Nissan', 'Toyota', 'Volkswagen', 'Volvo']


In [3]:
import pandas as pd
import os
from fuzzywuzzy import process, fuzz
from pathlib import Path

# ===== 1. BEIDE DATENSÄTZE LADEN =====
dataset1_path = r"C:\Users\pparr\Documents\Henallux\Henallux PP 2025_2026\Semester 2\Systèmes_intelligents\Dataset\Car Brand Classification Dataset\train"
dataset2_path = r"C:\Users\pparr\Documents\Henallux\Henallux PP 2025_2026\Semester 2\Systèmes_intelligents\Dataset\training-car\training_set"

def create_dataframe_from_folders(root_path):
    """Erstellt DataFrame aus Verzeichnisstruktur (Ordner = Marke, Bilder = Datei)"""
    data = []
    for brand_folder in os.listdir(root_path):
        folder_path = os.path.join(root_path, brand_folder)
        if os.path.isdir(folder_path):
            for image_file in os.listdir(folder_path):
                if image_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                    data.append({
                        'brand': brand_folder,
                        'image_file': image_file,
                        'full_path': os.path.join(folder_path, image_file),
                        'dataset_source': 'dataset1'
                    })
    return pd.DataFrame(data)

def create_dataframe_from_folders_v2(root_path, source_name='dataset'):
    """Alternative falls Dataset anders strukturiert ist"""
    data = []
    for brand_folder in os.listdir(root_path):
        folder_path = os.path.join(root_path, brand_folder)
        if os.path.isdir(folder_path):
            for image_file in os.listdir(folder_path):
                if image_file.lower().endswith(('.jpg', '.jpeg', '.png')):
                    data.append({
                        'brand': brand_folder,
                        'image_file': image_file,
                        'full_path': os.path.join(folder_path, image_file),
                        'dataset_source': source_name
                    })
    return pd.DataFrame(data)

# Laden
print("Lade Dataset 1...")
df1 = create_dataframe_from_folders(dataset1_path)
print(f"Dataset 1: {len(df1)} Bilder")
print(f"Marken: {df1['brand'].unique()}\n")

print("Lade Dataset 2...")
df2 = create_dataframe_from_folders_v2(dataset2_path, source_name='dataset2')
print(f"Dataset 2: {len(df2)} Bilder")
print(f"Marken: {df2['brand'].unique()}")

Lade Dataset 1...
Dataset 1: 11742 Bilder
Marken: <StringArray>
[        'Acura',  'Aston Martin',          'Audi',       'Bentley',
           'BMW',         'Buick',      'Cadillac',     'Chevrolet',
      'Chrysler',         'Dodge',          'FIAT',          'Ford',
           'GMC',         'Honda',       'Hyundai',      'INFINITI',
        'Jaguar',          'Jeep',           'Kia',    'Land Rover',
         'Lexus',       'Lincoln',         'Mazda', 'Mercedes-Benz',
          'MINI',    'Mitsubishi',        'Nissan',       'Porsche',
           'Ram',        'Subaru',        'Toyota',    'Volkswagen',
         'Volvo']
Length: 33, dtype: str

Lade Dataset 2...
Dataset 2: 13943 Bilder
Marken: <StringArray>
[       'Audi',     'Bentley',         'BMW',   'Chevrolet',    'Chrysler',
       'Dodge',     'Ferrari',        'Ford',         'GMC',       'Honda',
     'Hyundai', 'Lamborghini',    'Mercedes',      'Nissan',      'Toyota',
  'Volkswagen',       'Volvo']
Length: 17, dtype: 

In [4]:
# ===== 2. DATENSÄTZE ZUSAMMENFÜHREN =====
df_combined = pd.concat([df1, df2], ignore_index=True)
print(f"Kombinierte Datensätze: {len(df_combined)} Bilder")
print(f"Alle Marken VOR Bereinigung:\n{df_combined['brand'].value_counts()}\n")

Kombinierte Datensätze: 25685 Bilder
Alle Marken VOR Bereinigung:
brand
Hyundai          1391
Honda            1358
Volkswagen       1299
Nissan           1293
Audi             1285
Ford             1271
Toyota           1255
Chevrolet        1254
BMW              1252
Dodge            1197
Bentley          1083
Chrysler         1007
Mercedes          945
Volvo             771
GMC               749
Lamborghini       735
Ferrari           703
Buick             376
Subaru            376
Acura             375
INFINITI          373
Cadillac          372
FIAT              371
Porsche           370
Aston Martin      368
Ram               366
Jaguar            349
Jeep              349
Kia               349
Land Rover        349
Lexus             349
Lincoln           349
Mazda             349
Mercedes-Benz     349
MINI              349
Mitsubishi        349
Name: count, dtype: int64



In [5]:
# ===== 3. SCHREIBWEISEN NORMALISIEREN =====
def normalize_brand(name):
    """Vereinheitlicht Markennamen"""
    if pd.isna(name):
        return name
    
    name = str(name).lower().strip()
    
    # Häufige Variationen vereinheitlichen
    replacements = {
        "mercedes benz": "mercedes",
        "mercedes-benz": "mercedes",
        "benz": "mercedes",
        "bmw": "bmw",
        "volkswagen": "volkswagen",
        "vw": "volkswagen",
        "audi": "audi",
        "porsche": "porsche",
        "volvo": "volvo",
        "ford": "ford",
        "chevrolet": "chevrolet",
        "tesla": "tesla",
        "toyota": "toyota",
        "honda": "honda",
        "hyundai": "hyundai",
        "kia": "kia",
    }
    
    for alt, standard in replacements.items():
        if alt in name:
            return standard
    
    return name

# Anwenden
df_combined['brand_normalized'] = df_combined['brand'].apply(normalize_brand)

print("Markennamen VOR und NACH Normalisierung:")
print(df_combined[['brand', 'brand_normalized']].drop_duplicates().sort_values('brand'))

Markennamen VOR und NACH Normalisierung:
               brand brand_normalized
0              Acura            acura
375     Aston Martin     aston martin
743             Audi             audi
1441             BMW              bmw
1092         Bentley          bentley
1790           Buick            buick
2166        Cadillac         cadillac
2538       Chevrolet        chevrolet
2887        Chrysler         chrysler
3236           Dodge            dodge
3585            FIAT             fiat
16726        Ferrari          ferrari
3956            Ford             ford
4305             GMC              gmc
4654           Honda            honda
5003         Hyundai          hyundai
5352        INFINITI         infiniti
5725          Jaguar           jaguar
6074            Jeep             jeep
6423             Kia              kia
20802    Lamborghini      lamborghini
6772      Land Rover       land rover
7121           Lexus            lexus
7470         Lincoln          lincoln
8517     

In [6]:
# ===== 4. FUZZY MATCHING FÜR WEITERE VEREINHEITLICHUNG =====
# Findet ähnliche Namen (falls noch Variationen existieren)
unique_brands = df_combined['brand_normalized'].dropna().unique().tolist()

def fuzzy_match_brands(brand, choices=unique_brands, threshold=85):
    """Matcht Markennamen fuzzy bei Tippfehlern"""
    if pd.isna(brand):
        return brand
    
    match, score = process.extractOne(brand, choices, scorer=fuzz.token_set_ratio)
    return match if score >= threshold else brand

df_combined['brand_matched'] = df_combined['brand_normalized'].apply(fuzzy_match_brands)

print("Nach Fuzzy Matching:")
print(df_combined['brand_matched'].value_counts())

Nach Fuzzy Matching:
brand_matched
hyundai         1391
honda           1358
volkswagen      1299
mercedes        1294
nissan          1293
audi            1285
ford            1271
toyota          1255
chevrolet       1254
bmw             1252
dodge           1197
bentley         1083
chrysler        1007
volvo            771
gmc              749
lamborghini      735
ferrari          703
buick            376
subaru           376
acura            375
infiniti         373
cadillac         372
fiat             371
porsche          370
aston martin     368
ram              366
jaguar           349
jeep             349
kia              349
land rover       349
lexus            349
lincoln          349
mazda            349
mini             349
mitsubishi       349
Name: count, dtype: int64


In [7]:
# ===== 5. DUPLIKATE ERKENNEN UND ENTFERNEN =====
print(f"VORHER: {len(df_combined)} Zeilen")

# Exakte Duplikate
exact_dupes = df_combined.duplicated().sum()
print(f"Exakte Duplikate: {exact_dupes}")

# Duplikate basierend auf Datoeiname + Marke (same image in both folders)
dupes_by_file = df_combined.duplicated(subset=['brand_matched', 'image_file'], keep='first').sum()
print(f"Duplikate (same file in different folders): {dupes_by_file}")

# Entfernen
#df_combined = df_combined.drop_duplicates()
#df_combined = df_combined.drop_duplicates(subset=['brand_matched', 'image_file'], keep='first')

print(f"NACHHER: {len(df_combined)} Zeilen")
print(f"\nFINALE STATISTIK PRO MARKE:")
print(df_combined['brand_matched'].value_counts().sort_values(ascending=False))

VORHER: 25685 Zeilen
Exakte Duplikate: 0
Duplikate (same file in different folders): 197
NACHHER: 25685 Zeilen

FINALE STATISTIK PRO MARKE:
brand_matched
hyundai         1391
honda           1358
volkswagen      1299
mercedes        1294
nissan          1293
audi            1285
ford            1271
toyota          1255
chevrolet       1254
bmw             1252
dodge           1197
bentley         1083
chrysler        1007
volvo            771
gmc              749
lamborghini      735
ferrari          703
buick            376
subaru           376
acura            375
infiniti         373
cadillac         372
fiat             371
porsche          370
aston martin     368
ram              366
jaguar           349
jeep             349
kia              349
land rover       349
lexus            349
lincoln          349
mazda            349
mini             349
mitsubishi       349
Name: count, dtype: int64


In [9]:
# ===== 6. ERGEBNISSE SPEICHERN UND ANZEIGEN =====
# DataFrame für weitere Verarbeitung vorbereiten
df_final = df_combined[['brand_matched', 'image_file', 'full_path', 'dataset_source']].copy()
df_final.columns = ['brand', 'image_file', 'path', 'source']

# Speichern als CSV (optional)
output_path = r"C:\Users\pparr\Documents\Henallux\Henallux PP 2025_2026\Semester 2\Systèmes_intelligents\Deep_learning_projet_SN_PP\combined_dataset.csv"
df_final.to_csv(output_path, index=False)
print(f"✓ Gespeichert: {output_path}\n")

# Übersicht
print("FINALE DATASET ÜBERSICHT:")
print(f"Gesamte Bilder: {len(df_final)}")
print(f"Anzahl Marken: {df_final['brand'].nunique()}")
print(f"\nBilder pro Marke:")
print(df_final['brand'].value_counts())

# Erste Zeilen anzeigen
print(f"\nErste 10 Zeilen:")
print(df_final.head(10))

✓ Gespeichert: C:\Users\pparr\Documents\Henallux\Henallux PP 2025_2026\Semester 2\Systèmes_intelligents\Deep_learning_projet_SN_PP\combined_dataset.csv

FINALE DATASET ÜBERSICHT:
Gesamte Bilder: 25685
Anzahl Marken: 35

Bilder pro Marke:
brand
hyundai         1391
honda           1358
volkswagen      1299
mercedes        1294
nissan          1293
audi            1285
ford            1271
toyota          1255
chevrolet       1254
bmw             1252
dodge           1197
bentley         1083
chrysler        1007
volvo            771
gmc              749
lamborghini      735
ferrari          703
buick            376
subaru           376
acura            375
infiniti         373
cadillac         372
fiat             371
porsche          370
aston martin     368
ram              366
jaguar           349
jeep             349
kia              349
land rover       349
lexus            349
lincoln          349
mazda            349
mini             349
mitsubishi       349
Name: count, dtype: i